# 03 - Análisis Exploratorio de Datos (EDA)

Este notebook contiene el análisis exploratorio de los datasets limpios de ofertas laborales.

El objetivo de este análisis es entender la estructura general de los datos, detectar patrones relevantes y preparar la base para las visualizaciones, el análisis de sesgos y las conclusiones del proyecto.


## 1. Objetivo

El objetivo de este notebook es explorar los datasets limpios y comprender las principales características de las ofertas laborales analizadas.

Este análisis se centrará en:

- Estructura general de los datasets
- Valores nulos
- Puestos o roles profesionales
- Ubicaciones
- Modalidad de trabajo
- Nivel de experiencia o seniority
- Información salarial
- Skills y tecnologías
- Patrones principales e insights relevantes

## 2. Importación de librerías

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")


## 3. Carga de datasets limpios

En esta sección se cargan los datasets limpios disponibles en la carpeta `data/clean/`.

Estos archivos contienen información sobre ofertas laborales, roles tecnológicos y rankings de tecnologías 

In [7]:
import pandas as pd

jobs_clean = pd.read_csv("../data/clean/jobs_clean.csv")
tecno_jobs_clean = pd.read_csv("../data/clean/tecno_jobs_clean.csv")
jobs_all_clean = pd.read_csv("../data/clean/jobs_all_clean.csv")

stack_tech_columns_clean = pd.read_csv("../data/clean/stack_tech_columns_clean.csv")
technologies_clean_long_format = pd.read_csv("../data/clean/technologies_clean_long_format.csv")

technology_rankings = pd.read_csv("../data/clean/technology_rankings.csv")
technology_rankings_used = pd.read_csv("../data/clean/technology_rankings_used.csv")
technology_rankings_wanted = pd.read_csv("../data/clean/technology_rankings_wanted.csv")

## 4. Revisión inicial

En esta sección se revisa la estructura general de los datasets limpios disponibles.

Se comprobará el número de filas y columnas, los nombres de las variables y el contenido general de cada archivo. Esto nos permitirá entender qué información tenemos disponible antes de empezar el análisis exploratorio.

### 4.1 Dimensiones de los datasets

En esta sección se revisa el tamaño de cada dataset limpio.

El objetivo es conocer cuántas filas y columnas contiene cada archivo antes de decidir qué uso tendrá dentro del análisis exploratorio.

In [8]:
datasets = {
    "jobs_clean": jobs_clean,
    "tecno_jobs_clean": tecno_jobs_clean,
    "jobs_all_clean": jobs_all_clean,
    "stack_tech_columns_clean": stack_tech_columns_clean,
    "technologies_clean_long_format": technologies_clean_long_format,
    "technology_rankings": technology_rankings,
    "technology_rankings_used": technology_rankings_used,
    "technology_rankings_wanted": technology_rankings_wanted,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

jobs_clean: 944 rows, 13 columns
tecno_jobs_clean: 600 rows, 7 columns
jobs_all_clean: 1542 rows, 16 columns
stack_tech_columns_clean: 49191 rows, 13 columns
technologies_clean_long_format: 1176875 rows, 4 columns
technology_rankings: 372 rows, 4 columns
technology_rankings_used: 186 rows, 4 columns
technology_rankings_wanted: 186 rows, 4 columns


### 4.2 Resumen de los datasets

La siguiente tabla resume el número de filas, columnas y variables disponibles en cada dataset limpio.

Este resumen permite comparar rápidamente la estructura de los archivos y entender qué información aporta cada uno.

In [9]:
dataset_summary = pd.DataFrame({
    "dataset": list(datasets.keys()),
    "rows": [df.shape[0] for df in datasets.values()],
    "columns": [df.shape[1] for df in datasets.values()],
    "column_names": [", ".join(df.columns) for df in datasets.values()]
})

dataset_summary

,dataset,rows,columns,column_names
0,jobs_clean,944,13,"job_title, seniority_level, status, company, l..."
1,tecno_jobs_clean,600,7,"titulo, empresa, salario, ubicacion, tipo_de_t..."
2,jobs_all_clean,1542,16,"job_title, company, location, salary, job_type..."
3,stack_tech_columns_clean,49191,13,"ResponseId, LanguageHaveWorkedWith, LanguageWa..."
4,technologies_clean_long_format,1176875,4,"ResponseId, technology, category, type"
5,technology_rankings,372,4,"category, type, technology, count"
6,technology_rankings_used,186,4,"category, type, technology, count"
7,technology_rankings_wanted,186,4,"category, type, technology, count"


### 4.3 Revisión de puestos

En esta sección se revisan los puestos más frecuentes en los principales datasets de ofertas laborales.

Esto nos ayuda a entender si cada dataset está centrado en roles de datos o si incluye una variedad más amplia de puestos tecnológicos e IT.

In [10]:
jobs_all_clean["job_title"].value_counts().head(20)

job_title
data scientist                                                                               856
machine learning engineer                                                                     78
data engineer                                                                                  4
Project Manager                                                                                4
Electromecánico/a                                                                              4
Analista Programador/a .NET                                                                    3
Software Development Engineer                                                                  2
Programador/a PLC                                                                              2
Programador/a Cobol                                                                            2
IT Support Technician                                                                          2
Administrador de sis

In [11]:
tecno_jobs_clean["titulo"].value_counts().head(20)

titulo
Project Manager                                                                                 4
Electromecánico/a                                                                               4
Analista Programador/a .NET                                                                     3
Software Development Engineer                                                                   2
Programador/a PLC                                                                               2
Programador/a Cobol                                                                             2
IT Support Technician                                                                           2
Administrador de sistemas                                                                       2
Programador Java                                                                                2
Senior SAP SD Consultant                                                                        2
Técnico de si

In [12]:
jobs_clean["job_title"].value_counts().head(20)

job_title
data scientist               856
machine learning engineer     80
data engineer                  4
data analyst                   1
Name: count, dtype: int64

### 4.4 Selección inicial de datasets

La revisión de puestos muestra que los datasets disponibles tienen alcances diferentes.

`jobs_clean.csv` está muy concentrado en un número reducido de roles relacionados con datos, especialmente Data Scientist y Machine Learning Engineer. Por este motivo, es útil para analizar esos perfiles concretos, pero sus resultados no deben interpretarse como una representación equilibrada de todos los roles de datos.

`jobs_all_clean.csv` ofrece una visión más amplia, ya que incluye los roles relacionados con datos de `jobs_clean.csv` junto con otros puestos tecnológicos e IT. Este dataset puede ser útil para comparar, aunque también incluye puestos que no están directamente relacionados con datos.

`tecno_jobs_clean.csv` contiene ofertas tecnológicas, pero muchas de ellas corresponden a roles generales de IT o desarrollo de software, no necesariamente a perfiles específicos de datos.

Por este motivo, el EDA utilizará más de un dataset dependiendo de la pregunta que se quiera responder. El análisis evitará fusionar todos los datasets en un único modelo relacional, ya que no comparten una clave común fiable.

### 4.5 Tipos de datos

En esta sección se revisan los tipos de datos de los principales datasets de ofertas laborales.

Esto permite comprobar si las variables están correctamente interpretadas por pandas, por ejemplo si los salarios aparecen como valores numéricos, si las fechas están como texto o si las variables categóricas aparecen como tipo `object`.

In [13]:
main_job_datasets = {
    "jobs_clean": jobs_clean,
    "jobs_all_clean": jobs_all_clean,
    "tecno_jobs_clean": tecno_jobs_clean
}

for name, df in main_job_datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(df.dtypes)


jobs_clean
----------
job_title          str
seniority_level    str
status             str
company            str
location           str
post_date          str
headquarter        str
industry           str
ownership          str
company_size       str
revenue            str
salary             str
skills             str
dtype: object

jobs_all_clean
--------------
job_title                   str
company                     str
location                    str
salary                      str
job_type                    str
post_date                   str
link                        str
skills                      str
industry                    str
seniority_level             str
source_dataset              str
salary_clean            float64
location_clean              str
city_clean                  str
is_remote                  bool
salary_clean_outlier       bool
dtype: object

tecno_jobs_clean
----------------
titulo                  str
empresa                 str
salario         

### 4.6 Información general de los datasets

En esta sección se revisa, de forma resumida, el tipo de dato, el número de valores no nulos y el porcentaje de valores faltantes de cada columna en los principales datasets de ofertas laborales.

Este resumen nos permite detectar rápidamente qué variables pueden necesitar transformación o revisión antes de continuar con el análisis.

In [16]:
info_summary = []

for name, df in main_job_datasets.items():
    for column in df.columns:
        info_summary.append({
            "dataset": name,
            "column": column,
            "dtype": df[column].dtype,
            "non_null_values": df[column].notnull().sum(),
            "null_values": df[column].isnull().sum(),
            "null_percentage": round(df[column].isnull().mean() * 100, 2)
        })

info_summary = pd.DataFrame(info_summary)

info_summary

,dataset,column,dtype,non_null_values,null_values,null_percentage
0,jobs_clean,job_title,str,941,3,0.32
1,jobs_clean,seniority_level,str,884,60,6.36
2,jobs_clean,status,str,688,256,27.12
3,jobs_clean,company,str,944,0,0.00
4,jobs_clean,location,str,942,2,0.21
5,jobs_clean,post_date,str,944,0,0.00
6,jobs_clean,headquarter,str,944,0,0.00
7,jobs_clean,industry,str,944,0,0.00
8,jobs_clean,ownership,str,897,47,4.98
9,jobs_clean,company_size,str,944,0,0.00


La revisión general muestra que muchos campos relevantes están almacenados como texto (`object`), incluyendo variables como salario, fecha de publicación, ubicación, puesto y skills.

Esto es importante porque algunas columnas necesitarán transformaciones antes de realizar ciertos análisis. Por ejemplo, los salarios deberán revisarse o convertirse a formato numérico cuando sea posible, y las fechas de publicación deberán tratarse como fechas si se analizan temporalmente.

También se observa que algunos datasets presentan columnas con valores faltantes, por lo que será necesario revisar los nulos con más detalle en la siguiente sección.

### 4.7 Variables numéricas y categóricas

En esta sección se identifican las variables numéricas y categóricas de los datasets principales.

Esta separación es importante porque las variables numéricas se pueden analizar con estadísticas descriptivas y correlaciones, mientras que las variables categóricas se analizan con conteos, porcentajes y comparaciones por grupo.

In [17]:
for name, df in main_job_datasets.items():
    numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_columns = df.select_dtypes(include=["object", "string", "bool"]).columns.tolist()
    
    print(f"\n{name}")
    print("-" * len(name))
    print("Numeric columns:", numeric_columns)
    print("Categorical columns:", categorical_columns)


jobs_clean
----------
Numeric columns: []
Categorical columns: ['job_title', 'seniority_level', 'status', 'company', 'location', 'post_date', 'headquarter', 'industry', 'ownership', 'company_size', 'revenue', 'salary', 'skills']

jobs_all_clean
--------------
Numeric columns: ['salary_clean']
Categorical columns: ['job_title', 'company', 'location', 'salary', 'job_type', 'post_date', 'link', 'skills', 'industry', 'seniority_level', 'source_dataset', 'location_clean', 'city_clean', 'is_remote', 'salary_clean_outlier']

tecno_jobs_clean
----------------
Numeric columns: []
Categorical columns: ['titulo', 'empresa', 'salario', 'ubicacion', 'tipo_de_trabajo', 'fecha_de_publicacion', 'enlace']


La revisión de variables numéricas y categóricas muestra que la mayoría de las columnas están almacenadas como texto.

En `jobs_clean.csv` y `tecno_jobs_clean.csv` no se detectan variables numéricas, por lo que columnas como salario o fecha de publicación necesitarán tratamiento previo si se quieren analizar cuantitativamente.

En `jobs_all_clean.csv` sí aparece una variable numérica, `salary_clean`, que podrá utilizarse para análisis salariales iniciales.

Este resultado confirma que antes de realizar análisis estadísticos, correlaciones o visualizaciones numéricas será necesario revisar el formato de algunas variables.

### 4.8 Estadísticas descriptivas iniciales

En esta sección se calculan estadísticas descriptivas iniciales de los datasets principales.

Como la mayoría de las variables están almacenadas como texto, primero se revisarán las variables numéricas disponibles y después se analizarán algunas variables categóricas relevantes mediante conteos de valores.

In [18]:
for name, df in main_job_datasets.items():
    numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
    
    print(f"\n{name}")
    print("-" * len(name))
    
    if numeric_columns:
        display(df[numeric_columns].describe())
    else:
        print("No numeric columns available.")


jobs_clean
----------
No numeric columns available.

jobs_all_clean
--------------


,salary_clean
count,1.071000e+03
mean,1.204878e+05
std,1.248324e+05
min,7.055000e+03
25%,4.596825e+04
50%,1.193995e+05
75%,1.657745e+05
max,2.739979e+06



tecno_jobs_clean
----------------
No numeric columns available.


## 5. Valores nulos

En esta sección se analizarán los valores nulos presentes en los datasets principales.

El objetivo es identificar qué columnas tienen datos faltantes y cuánta información falta en cada una. Esto es especialmente importante en variables como salario, nivel de experiencia, ubicación, modalidad de trabajo o tipo de contrato, porque los valores nulos pueden afectar a las conclusiones del análisis.

## 6. Job title analysis

In this section, we will analyze the job titles included in the dataset.

We will review which roles appear most frequently and whether the dataset is focused on specific profiles, such as Data Analyst, Data Scientist, Software Developer, AI Engineer or other technology-related roles.

## 7. Company analysis

In this section, we will analyze the companies that appear in the dataset.

We will check which companies publish the highest number of job offers and whether the dataset is concentrated in a few companies or distributed across many different employers.

## 8. Location analysis

In this section, we will analyze the geographical distribution of the job offers.

We will check which cities or regions have the highest number of offers. This will help us understand whether the dataset is concentrated in specific locations, such as Madrid or Barcelona, or if it represents different areas of Spain more evenly.

## 9. Work modality analysis

In this section, we will analyze the work modality of the job offers.

We will review whether the offers are remote, hybrid or on-site. This analysis can help us understand current trends in the technology job market and how flexible the available positions are.

## 10. Contract type analysis

In this section, we will analyze the type of contract offered in the job postings.

We will review whether the offers are mainly full-time, part-time, internship, freelance, temporary or permanent positions, depending on the information available in the dataset.

## 11. Seniority analysis

In this section, we will analyze the seniority or experience level required in the job offers.

We will check whether companies are mostly looking for junior, mid-level or senior profiles. This will help us understand the accessibility of the market for candidates with different levels of experience.

## 12. Sector analysis

In this section, we will analyze the sectors represented in the dataset.

We will check whether the job offers are concentrated in specific sectors, such as consulting, finance, technology, education, healthcare or other industries.

## 13. Salary analysis

In this section, we will analyze salary information.

We will review salary ranges, average salaries and possible salary differences depending on variables such as seniority, location, work modality, contract type, sector or job title. We will also consider that some job offers may not include salary information.

## 14. Skills and technologies analysis

In this section, we will analyze the most demanded skills and technologies.

We will review which programming languages, tools, frameworks or technical skills appear most frequently in the job offers. This can help identify the technologies that are most valued in the current job market.

## 15. Job description analysis

In this section, we will analyze the job descriptions.

We will review whether the descriptions contain useful information about responsibilities, requirements, technologies, benefits or experience expectations. This section may also help identify repeated patterns in the language used in job offers.

## 16. Posting date analysis

In this section, we will analyze the publication dates of the job offers.

We will check when the offers were posted and whether there are periods with more activity. This can help us understand the temporal distribution of the dataset.

## 17. Main conclusions

In this section, we will summarize the main findings obtained from the exploratory data analysis.

The conclusions will be completed once the clean dataset is available and the analysis has been performed. This section will include the most relevant patterns found in the data and any important limitations detected during the analysis.